Our first notebook for the pipeline :)

In [1]:
#code goes here and in other cells
print("It's earthquake time")

It's earthquake time


Extract stage

In [2]:
import csv
import os
import time
from datetime import datetime, timedelta, timezone
import requests

"""What the code does is pull from the start time in month batches, and then writes to the csv per month, 
keeping a tab of months processed so if the pull gets cut it saves whatever it did so far"""


url = "https://earthquake.usgs.gov/fdsnws/event/1/"

startime = datetime(2006, 1, 1)
endtime = datetime.now(timezone.utc).replace(tzinfo=None)

minimum_magnitude = 2.5

request_limit = 10000

dataset = "earthquakes_2006_present.csv"
dataset_backup = "earthquakes_progress.txt"

features = ["id", "time", "mag", "magType", "place", "longitude", "latitude", "depth_km", "type", "status", "updated"]

"""sets up parameters to use in pulls"""
def build_params(start, end):
    p = {
        "starttime": start.strftime("%Y-%m-%dT%H:%M:%S"),
        "endtime": end.strftime("%Y-%m-%dT%H:%M:%S"),
        "orderby": "time-asc",
    }
    if minimum_magnitude is not None:
        p["minmagnitude"] = minimum_magnitude
    return p

"""does GET request and tries multiple times if pull fails"""
def get(method, params, fmt):
    for attempt in range(6):
        try:
            r = requests.get(url + method, params={**params, "format": fmt}, timeout=180)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            wait = 5
            print(f"request failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Giving up on {method} {params}")

""""takes the pulled json and makes it csv"""
def to_row(feature):
    p = feature["properties"]
    lon, lat, depth = (feature["geometry"]["coordinates"] + [None, None, None])[:3]
    iso = lambda ms: datetime.fromtimestamp(ms / 1000, timezone.utc).isoformat() if ms else None
    return {
        "id": feature["id"], "time": iso(p.get("time")), "mag": p.get("mag"),
        "magType": p.get("magType"), "place": p.get("place"),
        "longitude": lon, "latitude": lat, "depth_km": depth,
        "type": p.get("type"), "status": p.get("status"), "updated": iso(p.get("updated")),
    }

"""what fetch window does is just make sure we dont pull too much data from a single month"""
def fetch_window(start, end, writer, seen):
    params = build_params(start, end)
    n = int(get("count", params, "text").text.strip())
    if n == 0:
        return 0
    if n > request_limit:
        mid = start + (end - start) / 2
        print(f"{n} events in {start:%Y-%m-%d}..{end:%Y-%m-%d}, splitting")
        return fetch_window(start, mid, writer, seen) + fetch_window(mid, end, writer, seen)

    features = get("query", params, "geojson").json()["features"]
    added = 0
    for f in features:
        if f["id"] not in seen:
            seen.add(f["id"])
            writer.writerow(to_row(f))
            added += 1
    time.sleep(1)
    return added

""""This just sets month pull lengths"""
def month_windows(start, end):
    cur = start
    while cur < end:
        nxt = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        yield cur, min(nxt, end)
        cur = nxt

"""This reads all the saved rows and gets rid of rows that're potentially incomplete due to pull errors"""
def load_existing():
    if not os.path.exists(dataset):
        return set()
    with open(dataset, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    good = [r for r in rows if r.get("id") and r.get("updated") is not None]
    if len(good) != len(rows):
        print(f"Removing {len(rows) - len(good)} incomplete row(s) from {dataset}")
        with open(dataset, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=features)
            w.writeheader()
            w.writerows(good)
    return {r["id"] for r in good}

"""reloads already saved rows"""
def load_done_months():
    if not os.path.exists(dataset_backup):
        return set()
    with open(dataset_backup, encoding="utf-8") as fh:
        return {line.strip() for line in fh if line.strip()}


def main():
    seen = load_existing()
    done = load_done_months()
    if seen:
        print(f"Resuming: {len(seen)} events already saved, {len(done)} months finished")

    new_file = not os.path.exists(dataset)
    total = 0
    with open(dataset, "a", newline="", encoding="utf-8") as fh, \
         open(dataset_backup, "a", encoding="utf-8") as prog:
        writer = csv.DictWriter(fh, fieldnames=features)
        if new_file:
            writer.writeheader()
        for ws, we in month_windows(startime, endtime):
            key = f"{ws:%Y-%m}"
            if key in done:
                continue
            added = fetch_window(ws, we, writer, seen)
            fh.flush()
            os.fsync(fh.fileno())
            total += added
            print(f"{key}: {added:>6} new events  (this run: {total})")

            if we < endtime:
                prog.write(key + "\n")
                prog.flush()
    print(f"Done. {total} new events this run, {len(seen)} total in {dataset}")


if __name__ == "__main__":
    main()

2006-01:   2242 new events  (this run: 2242)
2006-02:   2178 new events  (this run: 4420)
2006-03:   2709 new events  (this run: 7129)
2006-04:   2843 new events  (this run: 9972)
2006-05:   2896 new events  (this run: 12868)
2006-06:   3444 new events  (this run: 16312)
2006-07:   2925 new events  (this run: 19237)
2006-08:   2671 new events  (this run: 21908)
2006-09:   2346 new events  (this run: 24254)
2006-10:   2564 new events  (this run: 26818)
2006-11:   3208 new events  (this run: 30026)
2006-12:   2477 new events  (this run: 32503)
2007-01:   2733 new events  (this run: 35236)
2007-02:   1988 new events  (this run: 37224)
2007-03:   2468 new events  (this run: 39692)
2007-04:   3196 new events  (this run: 42888)
2007-05:   2420 new events  (this run: 45308)
2007-06:   2332 new events  (this run: 47640)
2007-07:   2653 new events  (this run: 50293)
2007-08:   3113 new events  (this run: 53406)
2007-09:   2662 new events  (this run: 56068)
2007-10:   2676 new events  (this run:

Transform Stage

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
# if you want a clean csv uncomment these and run this file with the extracted csv in the same folder
#DATA_DIR = Path(__file__).resolve().parent
#RAW_CSV = DATA_DIR / "earthquakes_2006_present.csv"
#CLEAN_CSV = DATA_DIR / "earthquakes_clean.csv"
################################### cleaning functions #######################################


def standard_Letters(df, col):
    '''standardize txt strip spaces make lowercase takes in df and the column you clean'''
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for c in col:
        df[c] =(df[c].astype("string")
                .str.strip()
                .str.lower()
                .str.replace(r"\s+", " ", regex=True)
                .replace("", pd.NA))
    return df

def convert_Datetime(df):
    '''convert time and updated to date time if updated is missing fill in with time'''
    df = df.copy()
    for c in ("time","updated"):
        df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    df["updated"] = df["updated"].fillna(df["time"])
    return df

def convert_Numeric(df):
    '''convert columns to numeric''' 
    df = df.copy()
    for c in ("mag","longitude","latitude","depth_km"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def fill_missing(df,col):
    '''takes df and column name replaces all na with str unknown'''
    df = df.copy()
    for c in col:
        df[c] = df[c].fillna("unknown")
    return df

def drop_dupes(df):
    '''drops all duplicates keeps just the most recent row for each id'''
    df = df.copy()
    df = df.sort_values("updated",na_position="first",kind="mergesort")
    df = df.drop_duplicates(subset=["id"], keep="last").reset_index(drop=True)
    return df

def range_check(df):
    '''turns all impossible values as nan make sure to do before fill missing'''
    df = df.copy()
    df.loc[~df["latitude"].between(-90, 90), "latitude"] = np.nan
    df.loc[~df["longitude"].between(-180, 180), "longitude"] = np.nan
    df.loc[~df["mag"].between(-2, 10), "mag"] = np.nan
    return df

def drop_reqs(df,col=("id", "time", "latitude", "longitude")):
    '''drop rows with missing data we need'''
    df = df.copy()
    df = df.dropna(subset= list(col))
    df = df.reset_index(drop=True)
    return df

def add_time(df):
    '''adds precise time columns'''
    df = df.copy()
    df["year"] = df["time"].dt.year
    df["month"] = df["time"].dt.month
    df["event_date"] = df["time"].dt.date
    return df

def filter_earth(df):
    '''drop all the rows taht arent earthquakes'''
    df = df.copy()
    df = df[df["type"].isin(["earthquake"])]
    return df.reset_index(drop= True)

def add_region(df):
    '''extract a usable region out of the txt in place column(txt after ,)'''
    df = df.copy()
    region = df["place"].astype("string").str.split(",").str[-1].str.strip()
    df["region"] = region.replace("", pd.NA).fillna("unknown")
    return df

def fill_depth(df):
    '''fill missing depths with region medians'''
    df = df.copy()
    df["depth_imputed"] = df["depth_km"].isna()
    regional = df.groupby("region")["depth_km"].transform("median")
    df["depth_km"] = df["depth_km"].fillna(regional)
    df["depth_km"] = df["depth_km"].fillna(df["depth_km"].median())
    return df



################################# cleaning stuff ######################################
text_cols = ["magtype", "type", "status"]
label_cols = ["magtype", "type", "status", "place"]
def clean(df):
    stats = {"received": len(df)}

    #standarize stuff fix type
    df = standard_Letters(df, text_cols)
    df = convert_Datetime(df)
    df = convert_Numeric(df)
    df = range_check(df)

    #drop the columns count #
    n = len(df)
    df = drop_reqs(df)
    stats["dropped_required"] = n - len(df)

    n = len(df)
    df = drop_dupes(df)
    stats["duplicates"] = n - len(df)

    n = len(df)
    df = filter_earth(df)
    stats["non_earthquake"] = n - len(df)


    #fill in missing stuff
    for c in label_cols:
        stats["missing_" + c] = int(df[c].isna().sum())
    stats["missing_mag"] = int(df["mag"].isna().sum())

    df = fill_missing(df, label_cols)
    df = add_region(df)

    stats["missing_depth_km"] = int(df["depth_km"].isna().sum())
    df = fill_depth(df)

    df = add_time(df)

    

    stats["after_cleaning"] = len(df)
    return df, stats


missing_treatment = {
    "magtype":  "filled with 'unknown'",
    "type":     "filled with 'unknown'",
    "status":   "filled with 'unknown'",
    "place":    "filled with 'unknown'",
    "mag":      "left null",
    "depth_km": "filled with regional median",
}
#####################################################  print the summary  ##################################

def print_summary(stats, since=None):
    '''Print the data quality summary required for every ETL run.'''
    window = f" (incremental since {since})" if since else " (full pull)"

    print(f"\nETL run {datetime.now():%Y-%m-%d %H:%M}")
    print(f"received from API: {stats['received']:,} rows{window}")
    print(f"after cleaning: {stats['after_cleaning']:,} rows, "
          f"{stats['duplicates']:,} duplicates removed")

    parts = []
    for c, how in missing_treatment.items():
        n = stats.get("missing_" + c, 0)
        parts.append(f"{c} {n:,} {how}")
    print("missing values: " + ", ".join(parts))

    print(f"rows dropped: {stats['dropped_required']:,} missing id/time/coords, "
          f"{stats['non_earthquake']:,} non-earthquake events")

######################################  main function #########################################
if __name__ == "__main__":
    raw = pd.read_csv("earthquakes_2006_present.csv", dtype=str)
    cleaned, stats = clean(raw)
    print_summary(stats)
    cleaned.to_csv("earthquakes_clean.csv", index=False)


ETL run 2026-11-11 09:05
received from API: 568,212 rows (full pull)
after cleaning: 545,891 rows, 0 duplicates removed
missing values: magtype 1 filled with 'unknown', type 0 filled with 'unknown', status 0 filled with 'unknown', place 0 filled with 'unknown', mag 0 left null, depth_km 0 filled with regional median
rows dropped: 17,270 missing id/time/coords, 5,051 non-earthquake events
